# V7 note

This notebook uses jointly classified V7 shock episodes. `ambiguous` episodes remain available for KM analysis but are excluded from KM–alpha comparisons because no unique responding market can be assigned.

# Saved V5 Kaplan–Meier results and VECM comparison

This notebook assumes the V5 Kaplan–Meier processing has already completed and the monthly KM grids are saved.

It therefore starts from:

```python
load_monthly_km(...)
```

and does **not** rerun raw trade pulls, event extraction, or monthly aggregation.

The workflow:

1. loads the saved monthly KM grids for ETH linear and inverse perpetuals;
2. calculates the KM-estimated percentage reaching 90% basis resolution within 10 seconds;
3. graphs the monthly percentages with confidence intervals;
4. reads the saved VECM results using the same `read_files(...)` calls as the attached Hasbrouck notebook;
5. converts hourly VECM estimates to monthly summaries;
6. compares KM resolution with the responding market's error-correction coefficient and with information leadership;
7. repeats the comparison across VECM aggregation latencies;
8. saves analysis-ready CSV outputs.

## 1. Imports

In [ ]:
from pathlib import Path
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from scipy.stats import pearsonr, spearmanr

from survival_analysis_basis_pipeline_v7 import load_monthly_km
from vecm_hasbrouck3 import *

pd.set_option("display.max_columns", 200)

## 2. Configuration

Set the four KM checkpoint directories to the directories already containing the saved V5 `km/` files.

`load_monthly_km(checkpoint_dir, first=...)` automatically reads the small saved monthly KM CSVs from `checkpoint_dir / "km"`.

In [ ]:
# Existing V5 checkpoint directories
ETH_UM_DIR = Path("sa_eth_um")
ETH_CM_DIR = Path("sa_eth_cm")

KM_RESOLUTION_PCT = 90
KM_HORIZON_SECONDS = 10

# VECM settings copied from the attached notebook
VECM_START = datetime.datetime(2021, 1, 1)
VECM_END = datetime.datetime(2025, 12, 31)

VECM_AGGS = ["1s", "500ms", "200ms", "100ms", "50ms", "10ms"]
VECM_LAGS = [10]
VECM_INTERVAL = "1H"

VECM_PREFIX = "hasbrouck3_eth_alignedv2"
VECM_UM_FOLDER = "vecm_hasbrouck3_eth_um"
VECM_CM_FOLDER = "vecm_hasbrouck3_eth_cm"

VECM_MONTHLY_AGGREGATION = "median"

## 3. Load the saved monthly KM grids

This is the only KM input step. No raw trades are downloaded or processed.

In [ ]:
eth_um_km_spot = load_monthly_km(ETH_UM_DIR, first="all")
eth_um_km_perp = load_monthly_km(ETH_UM_DIR, first="all")

eth_cm_km_spot = load_monthly_km(ETH_CM_DIR, first="all")
eth_cm_km_perp = load_monthly_km(ETH_CM_DIR, first="all")

print("ETH linear spot KM rows:", len(eth_um_km_spot))
print("ETH linear perp KM rows:", len(eth_um_km_perp))
print("ETH inverse spot KM rows:", len(eth_cm_km_spot))
print("ETH inverse perp KM rows:", len(eth_cm_km_perp))

In [ ]:
def label_km_frame(frame, contract_type, first):
    output = frame.copy()
    output["contract_type"] = contract_type

    if "first" not in output.columns:
        output["first"] = first

    return output


km_all = pd.concat(
    [
        label_km_frame(eth_um_km_spot, "linear", "spot"),
        label_km_frame(eth_um_km_perp, "linear", "perp"),
        label_km_frame(eth_cm_km_spot, "inverse", "spot"),
        label_km_frame(eth_cm_km_perp, "inverse", "perp"),
    ],
    ignore_index=True,
)

km_all.head()

In [ ]:
def extract_resolution_probability(
    km_grid,
    horizon_seconds=10.0,
    resolution_pct=90,
):
    required = {
        "period",
        "time_s",
        "survival",
        "ci_lower",
        "ci_upper",
    }
    missing = required.difference(km_grid.columns)

    if missing:
        raise ValueError(f"KM grid is missing columns: {sorted(missing)}")

    data = km_grid.copy()

    if "resolution_pct" in data.columns:
        data = data[
            data["resolution_pct"].astype(int).eq(int(resolution_pct))
        ].copy()

    data["time_s"] = pd.to_numeric(data["time_s"], errors="coerce")
    data = data.dropna(
        subset=["period", "time_s", "survival", "ci_lower", "ci_upper"]
    )

    group_columns = [
        column
        for column in [
            "period",
            "contract_type",
            "first",
            "shock_significance",
            "resolution_pct",
        ]
        if column in data.columns
    ]

    rows = []

    for group_key, group in data.groupby(
        group_columns,
        dropna=False,
        observed=True,
    ):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        group = group.sort_values("time_s")

        # KM is a right-continuous step function. Use the last saved estimate
        # at or before the requested horizon.
        eligible = group[group["time_s"] <= horizon_seconds]

        if eligible.empty:
            continue

        horizon_row = eligible.iloc[-1]
        survival = float(horizon_row["survival"])
        survival_lower = float(horizon_row["ci_lower"])
        survival_upper = float(horizon_row["ci_upper"])

        result = dict(zip(group_columns, group_key))
        result.update(
            {
                "horizon_seconds": float(horizon_seconds),
                "survival_at_horizon": survival,
                "resolved_share": 1.0 - survival,
                "resolved_ci_lower": 1.0 - survival_upper,
                "resolved_ci_upper": 1.0 - survival_lower,
                "resolved_percent": 100.0 * (1.0 - survival),
                "resolved_ci_lower_percent": 100.0 * (1.0 - survival_upper),
                "resolved_ci_upper_percent": 100.0 * (1.0 - survival_lower),
            }
        )

        for column in ["n_episodes", "n_resolved", "censoring_share"]:
            if column in group.columns:
                result[column] = horizon_row[column]

        rows.append(result)

    output = pd.DataFrame(rows)

    if not output.empty:
        output["period"] = output["period"].astype(str)
        output["month"] = pd.to_datetime(
            output["period"],
            format="%Y-%m",
            errors="coerce",
        )

    return output

In [ ]:
km_10s = extract_resolution_probability(
    km_all,
    horizon_seconds=KM_HORIZON_SECONDS,
    resolution_pct=KM_RESOLUTION_PCT,
)

km_10s.sort_values(
    ["contract_type", "first", "month"]
).head(12)

## 5. Graph the monthly percentage resolved within 10 seconds

In [ ]:
def plot_resolution_within_horizon(
    resolution_summary,
    horizon_seconds=10,
    resolution_pct=90,
):
    data = resolution_summary.dropna(
        subset=[
            "month",
            "resolved_percent",
            "resolved_ci_lower_percent",
            "resolved_ci_upper_percent",
        ]
    ).copy()

    contract_types = [
        contract
        for contract in ["linear", "inverse"]
        if contract in data["contract_type"].unique()
    ]

    fig, axes = plt.subplots(
        len(contract_types),
        1,
        figsize=(12, 4.5 * len(contract_types)),
        sharex=True,
        squeeze=False,
    )

    for axis, contract_type in zip(axes[:, 0], contract_types):
        contract_data = data[
            data["contract_type"].eq(contract_type)
        ]

        for first, group in contract_data.groupby("first", observed=True):
            group = group.sort_values("month")

            line = axis.plot(
                group["month"],
                group["resolved_percent"],
                marker="o",
                markersize=3,
                linewidth=1.5,
                label=f"{first}-originating shocks",
            )[0]

            axis.fill_between(
                group["month"],
                group["resolved_ci_lower_percent"],
                group["resolved_ci_upper_percent"],
                alpha=0.15,
                color=line.get_color(),
            )

        axis.set_title(f"{contract_type.capitalize()} perpetual")
        axis.set_ylabel(f"Resolved within {horizon_seconds:g}s (%)")
        axis.set_ylim(0, 100)
        axis.grid(axis="y", alpha=0.3)
        axis.legend()

    axes[-1, 0].set_xlabel("Month")
    axes[-1, 0].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    fig.suptitle(
        f"KM-estimated share reaching {resolution_pct}% basis resolution "
        f"within {horizon_seconds:g} seconds",
        y=1.01,
    )

    fig.tight_layout()
    plt.show()

In [ ]:
plot_resolution_within_horizon(
    km_10s,
    horizon_seconds=KM_HORIZON_SECONDS,
    resolution_pct=KM_RESOLUTION_PCT,
)

In [ ]:
linear_res_dict = read_files(
    VECM_START,
    VECM_END,
    VECM_AGGS,
    VECM_INTERVAL,
    VECM_LAGS,
    "um",
    VECM_PREFIX,
    VECM_UM_FOLDER,
)

inverse_res_dict = read_files(
    VECM_START,
    VECM_END,
    VECM_AGGS,
    VECM_INTERVAL,
    VECM_LAGS,
    "cm",
    VECM_PREFIX,
    VECM_CM_FOLDER,
)

print("Linear result keys:", sorted(linear_res_dict.keys()))
print("Inverse result keys:", sorted(inverse_res_dict.keys()))

## 7. Convert hourly VECM results to monthly values

The VECM files contain one row for `log_midpoint_spot` and one row for `log_midpoint_perp` in each hourly interval. The function below:

1. removes duplicate interval-series estimates, preferring the row with more observations;
2. pivots spot and perp estimates into one row;
3. takes the monthly median or mean.

The median is the default because hourly alpha and share estimates can contain extreme values.

## 6. Read all saved VECM latency–lag combinations

The VECM dictionaries are keyed by `(latency, lag_base)`. This workflow uses every shared combination in the linear and inverse result dictionaries rather than selecting a primary case.

In [ ]:
linear_res_dict = read_files(
    VECM_START, VECM_END, VECM_AGGS, VECM_INTERVAL,
    VECM_LAGS, "um", VECM_PREFIX, VECM_UM_FOLDER,
)

inverse_res_dict = read_files(
    VECM_START, VECM_END, VECM_AGGS, VECM_INTERVAL,
    VECM_LAGS, "cm", VECM_PREFIX, VECM_CM_FOLDER,
)

available_combinations = sorted(
    set(linear_res_dict).intersection(inverse_res_dict)
)

if not available_combinations:
    raise RuntimeError("No shared latency–lag combinations were found.")

available_combinations

## 7. Convert one VECM combination to monthly values

In [ ]:
def prepare_monthly_vecm_results(
    results_dict,
    contract_type,
    latency,
    lag_base=10,
    monthly_aggregation="median",
):
    key = (latency, lag_base)

    if key not in results_dict:
        raise KeyError(
            f"{key} was not found. Available keys: "
            f"{sorted(results_dict.keys())}"
        )

    raw = results_dict[key].copy()
    raw["interval"] = pd.to_datetime(raw["interval"], errors="coerce")
    raw = raw.dropna(subset=["interval"])

    expected_series = {
        "log_midpoint_spot",
        "log_midpoint_perp",
    }
    actual_series = set(raw["series"].dropna().astype(str).unique())
    missing_series = expected_series.difference(actual_series)

    if missing_series:
        raise ValueError(
            f"Missing VECM series: {sorted(missing_series)}. "
            f"Observed series include: {sorted(actual_series)}"
        )

    duplicate_keys = ["interval", "series"]

    if "n_obs" in raw.columns:
        raw = (
            raw.sort_values(
                duplicate_keys + ["n_obs"],
                ascending=[True, True, False],
            )
            .drop_duplicates(duplicate_keys, keep="first")
        )
    else:
        raw = raw.drop_duplicates(duplicate_keys, keep="first")

    value_columns = [
        column
        for column in [
            "alpha",
            "CS",
            "HIS_lower",
            "HIS_upper",
            "HIS_mid",
            "ILS_mid",
            "n_obs",
        ]
        if column in raw.columns
    ]

    wide = raw.pivot_table(
        index="interval",
        columns="series",
        values=value_columns,
        aggfunc="first",
    )

    wide.columns = [
        f"{metric}_{series.replace('log_midpoint_', '')}"
        for metric, series in wide.columns
    ]
    wide = wide.reset_index()

    wide["month"] = wide["interval"].dt.to_period("M").dt.to_timestamp()
    wide["period"] = wide["month"].dt.strftime("%Y-%m")
    wide["contract_type"] = contract_type
    wide["latency"] = latency
    wide["lag_base"] = lag_base

    identifier_columns = {
        "interval",
        "month",
        "period",
        "contract_type",
        "latency",
        "lag_base",
    }
    numeric_columns = [
        column
        for column in wide.columns
        if column not in identifier_columns
    ]

    grouped = wide.groupby(
        ["period", "month", "contract_type", "latency", "lag_base"],
        as_index=False,
        observed=True,
    )[numeric_columns]

    if monthly_aggregation == "median":
        monthly = grouped.median()
    elif monthly_aggregation == "mean":
        monthly = grouped.mean()
    else:
        raise ValueError(
            "monthly_aggregation must be 'median' or 'mean'."
        )

    if "ILS_mid_spot" in monthly.columns:
        monthly["ILS_spot"] = monthly["ILS_mid_spot"]

    if "ILS_mid_perp" in monthly.columns:
        monthly["ILS_perp"] = monthly["ILS_mid_perp"]

    if "alpha_spot" in monthly.columns:
        monthly["abs_alpha_spot"] = monthly["alpha_spot"].abs()

    if "alpha_perp" in monthly.columns:
        monthly["abs_alpha_perp"] = monthly["alpha_perp"].abs()

    return monthly

## 8. Build monthly VECM results for every combination

In [ ]:
vecm_frames = []

for latency, lag_base in available_combinations:
    vecm_frames.append(
        prepare_monthly_vecm_results(
            linear_res_dict,
            contract_type="linear",
            latency=latency,
            lag_base=lag_base,
            monthly_aggregation=VECM_MONTHLY_AGGREGATION,
        )
    )
    vecm_frames.append(
        prepare_monthly_vecm_results(
            inverse_res_dict,
            contract_type="inverse",
            latency=latency,
            lag_base=lag_base,
            monthly_aggregation=VECM_MONTHLY_AGGREGATION,
        )
    )

vecm_monthly_all = pd.concat(vecm_frames, ignore_index=True)

vecm_monthly_all[
    ["contract_type", "latency", "lag_base"]
].drop_duplicates().sort_values(
    ["lag_base", "latency", "contract_type"]
)

## 9. Merge the fixed KM outcome with every VECM combination

In [ ]:
km_10s_origin_specific = km_10s[km_10s["first"].isin(["spot", "perp"])].copy()

comparison_all = km_10s_origin_specific.merge(
    vecm_monthly_all,
    on=["period", "month", "contract_type"],
    how="inner",
)

comparison_all["responding_alpha"] = np.where(
    comparison_all["first"].eq("spot"),
    comparison_all["alpha_perp"],
    comparison_all["alpha_spot"],
)

comparison_all["abs_responding_alpha"] = (
    comparison_all["responding_alpha"].abs()
)

if {"ILS_spot", "ILS_perp"}.issubset(comparison_all.columns):
    comparison_all["origin_market_ILS"] = np.where(
        comparison_all["first"].eq("spot"),
        comparison_all["ILS_spot"],
        comparison_all["ILS_perp"],
    )

comparison_all.head()

## 10. Plot every latency–lag combination

In [ ]:
def plot_one_vecm_combination(comparison, latency, lag_base):
    data = comparison[
        comparison["latency"].eq(latency)
        & comparison["lag_base"].eq(lag_base)
    ].dropna(
        subset=["abs_responding_alpha", "resolved_percent"]
    ).copy()

    if data.empty:
        print(f"No merged data for {(latency, lag_base)}")
        return

    contract_types = [
        c for c in ["linear", "inverse"]
        if c in data["contract_type"].unique()
    ]

    fig, axes = plt.subplots(
        1, len(contract_types),
        figsize=(6 * len(contract_types), 5),
        squeeze=False,
    )

    for axis, contract_type in zip(axes[0], contract_types):
        contract_data = data[
            data["contract_type"].eq(contract_type)
        ]

        for first, group in contract_data.groupby("first", observed=True):
            axis.scatter(
                group["abs_responding_alpha"],
                group["resolved_percent"],
                alpha=0.65,
                label=f"{first}-originating",
            )

            valid = group[
                ["abs_responding_alpha", "resolved_percent"]
            ].dropna()

            if len(valid) >= 3:
                slope, intercept = np.polyfit(
                    valid["abs_responding_alpha"],
                    valid["resolved_percent"],
                    1,
                )
                x = np.linspace(
                    valid["abs_responding_alpha"].min(),
                    valid["abs_responding_alpha"].max(),
                    100,
                )
                axis.plot(x, intercept + slope * x)

        axis.set_title(f"{contract_type.capitalize()} perpetual")
        axis.set_xlabel("Absolute responding-market alpha")
        axis.set_ylabel("Resolved within 10 seconds (%)")
        axis.set_ylim(0, 100)
        axis.grid(alpha=0.3)
        axis.legend()

    fig.suptitle(
        "Basis resolution versus VECM error correction "
        f"({latency}, lag base {lag_base})",
        y=1.02,
    )
    fig.tight_layout()
    plt.show()


for latency, lag_base in available_combinations:
    plot_one_vecm_combination(
        comparison_all,
        latency,
        lag_base,
    )

## 11. Correlations for every combination

In [ ]:
def correlation_summary_all(comparison):
    rows = []

    for keys, group in comparison.groupby(
        ["contract_type", "first", "latency", "lag_base"],
        observed=True,
    ):
        valid = group[
            ["abs_responding_alpha", "resolved_share"]
        ].replace([np.inf, -np.inf], np.nan).dropna()

        if len(valid) < 3:
            continue

        pearson_r, pearson_p = pearsonr(
            valid["abs_responding_alpha"],
            valid["resolved_share"],
        )
        spearman_rho, spearman_p = spearmanr(
            valid["abs_responding_alpha"],
            valid["resolved_share"],
        )

        rows.append({
            "contract_type": keys[0],
            "first": keys[1],
            "latency": keys[2],
            "lag_base": keys[3],
            "n_months": len(valid),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
        })

    return pd.DataFrame(rows)


correlations_all = correlation_summary_all(comparison_all)

correlations_all.sort_values(
    ["contract_type", "first", "lag_base", "latency"]
)

## 12. Plot how the relationship changes across latency

In [ ]:
def latency_to_seconds(latency):
    latency = str(latency).lower().strip()
    if latency.endswith("ms"):
        return float(latency[:-2]) / 1000
    if latency.endswith("s"):
        return float(latency[:-1])
    raise ValueError(f"Unsupported latency: {latency}")


def plot_correlations_across_latency(correlations):
    data = correlations.copy()
    data["latency_seconds"] = data["latency"].map(
        latency_to_seconds
    )

    fig, ax = plt.subplots(figsize=(11, 6))

    for keys, group in data.groupby(
        ["contract_type", "first", "lag_base"],
        observed=True,
    ):
        group = group.sort_values("latency_seconds")
        ax.plot(
            group["latency_seconds"],
            group["spearman_rho"],
            marker="o",
            label=(
                f"{keys[0]}, {keys[1]}-originating, "
                f"lag base {keys[2]}"
            ),
        )

    ax.axhline(0, linestyle="--", alpha=0.6)
    ax.set_xscale("log")
    ax.set_xlabel("VECM aggregation latency (seconds, log scale)")
    ax.set_ylabel("Spearman correlation")
    ax.set_title(
        "KM–VECM correspondence across aggregation latencies"
    )
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    plt.show()


plot_correlations_across_latency(correlations_all)

## 13. Heatmaps across latency and lag base

In [ ]:
def plot_correlation_heatmaps(correlations):
    data = correlations.copy()
    latency_order = sorted(
        data["latency"].unique(),
        key=latency_to_seconds,
    )
    lag_order = sorted(data["lag_base"].unique())

    groups = list(
        data[["contract_type", "first"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )

    fig, axes = plt.subplots(
        len(groups), 1,
        figsize=(9, 4 * len(groups)),
        squeeze=False,
    )

    for axis, (contract_type, first) in zip(
        axes[:, 0], groups
    ):
        subset = data[
            data["contract_type"].eq(contract_type)
            & data["first"].eq(first)
        ]

        pivot = subset.pivot(
            index="lag_base",
            columns="latency",
            values="spearman_rho",
        ).reindex(
            index=lag_order,
            columns=latency_order,
        )

        image = axis.imshow(
            pivot.to_numpy(dtype=float),
            aspect="auto",
            vmin=-1,
            vmax=1,
        )

        axis.set_title(
            f"{contract_type.capitalize()}, "
            f"{first}-originating shocks"
        )
        axis.set_xlabel("Latency")
        axis.set_ylabel("Lag base")
        axis.set_xticks(range(len(pivot.columns)))
        axis.set_xticklabels(pivot.columns)
        axis.set_yticks(range(len(pivot.index)))
        axis.set_yticklabels(pivot.index)

        for r in range(len(pivot.index)):
            for c in range(len(pivot.columns)):
                value = pivot.iloc[r, c]
                if pd.notna(value):
                    axis.text(c, r, f"{value:.2f}",
                              ha="center", va="center")

    fig.colorbar(
        image,
        ax=axes[:, 0].tolist(),
        label="Spearman correlation",
        shrink=0.8,
    )
    plt.show()


plot_correlation_heatmaps(correlations_all)

## 14. Check alpha-sign validity

Taking an absolute value can make a wrong-sign coefficient appear to represent strong error correction. This diagnostic reports the share of months with the conventional responding-market sign:

- spot-originating shock: `alpha_perp > 0`;
- perp-originating shock: `alpha_spot < 0`.

In [ ]:
comparison_all["conventional_responding_sign"] = np.where(
    comparison_all["first"].eq("spot"),
    comparison_all["alpha_perp"].gt(0),
    comparison_all["alpha_spot"].lt(0),
)

sign_summary = (
    comparison_all.groupby(
        ["contract_type", "first", "latency", "lag_base"],
        observed=True,
    )["conventional_responding_sign"]
    .agg(
        conventional_sign_share="mean",
        n_months="size",
    )
    .reset_index()
)

sign_summary.sort_values(
    ["contract_type", "first", "lag_base", "latency"]
)

## 15. Correlations using conventional-sign months only

In [ ]:
comparison_conventional = comparison_all[
    comparison_all["conventional_responding_sign"]
].copy()

correlations_conventional = correlation_summary_all(
    comparison_conventional
)

correlations_conventional.sort_values(
    ["contract_type", "first", "lag_base", "latency"]
)

## 16. Save outputs

In [ ]:
OUTPUT_DIR = Path(
    "sa_results/km_v5_vecm_all_latency_lag_combinations"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

km_10s.to_csv(
    OUTPUT_DIR / "km_resolved_within_10s.csv",
    index=False,
)
vecm_monthly_all.to_csv(
    OUTPUT_DIR / "vecm_monthly_all_combinations.csv",
    index=False,
)
comparison_all.to_csv(
    OUTPUT_DIR / "km_vecm_comparison_all_combinations.csv",
    index=False,
)
correlations_all.to_csv(
    OUTPUT_DIR / "km_vecm_correlations_all_combinations.csv",
    index=False,
)
sign_summary.to_csv(
    OUTPUT_DIR / "responding_alpha_sign_summary.csv",
    index=False,
)
correlations_conventional.to_csv(
    OUTPUT_DIR / "km_vecm_correlations_conventional_sign_only.csv",
    index=False,
)

print("Saved outputs to:", OUTPUT_DIR.resolve())

## Interpretation

This analysis holds the KM measure fixed and changes the VECM latency and lag specification. The cross-latency Spearman-correlation plot shows whether the within-specification ranking of months by VECM adjustment corresponds to the ranking by realized 10-second basis resolution.

Raw alpha magnitudes should not be compared directly across latencies because each alpha is measured per model observation.